In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Form_molsysmt_MolSys)=
# MolSys

*Native in-memory container combining molecular topology and structural trajectories.*

- **Technical Form Name:** `molsysmt.MolSys`
- **Form Type:** `class`
- **Origin / Ecosystem:** MolSysMT Native Object

:::{versionadded} 1.0.0
:::

:::{admonition} API Documentation
:class: info

- Class reference: {class}`molsysmt.MolSys` (alias of {class}`molsysmt.native.molsys.MolSys`)
- Form adapter module: {mod}`molsysmt.form.molsysmt_MolSys`
:::

## Overview

`molsysmt.MolSys` is MolSysMT's primary native in-memory representation for complete molecular systems. It pairs a **Topology** object ({class}`molsysmt.Topology`)—which defines the chemical graph, hierarchical element partitioning (atoms, groups/residues, components, molecules, entities, chains), and covalent bonds—with a **Structures** object ({class}`molsysmt.Structures`)—which stores 3D coordinates across one or multiple frames, unit cell box dimensions, timestamps, velocities, B-factors, and occupancy data.

### Key Characteristics:
- **In-Memory & Mutable:** Direct, high-performance programmatic access and modifications to coordinates, topology, and metadata.
- **Multi-Frame Support:** Seamlessly handles single static conformations or large multi-frame trajectories.
- **Full Tier-1 Compatibility:** Serves as the central conversion hub across all supported third-party classes, file formats, and chemical string representations.


## Supported Attributes

`molsysmt.MolSys` is a full-featured container supporting up to 115 topological, structural, and mechanical attributes:

- **Topological attributes:** Atom names/types/elements, group names/types, component IDs, molecule types, chain IDs, entity partitions, covalent bond connectivity, formal charges.
- **Structural attributes:** 3D Cartesian coordinates (`(n_structures, n_atoms, 3)`), unit cell box vectors and angles, structure IDs, timestamps, B-factors, occupancies, bioassemblies.
- **Mechanical & PhysChem attributes:** Potential/kinetic energies, temperature, force fields, non-bonded methods, implicit solvent parameters.

Let's verify the supported attributes programmatically:


In [2]:
import molsysmt as msm

form_name = 'molsysmt.MolSys'

# Check total number of supported attributes
attrs = msm.form.get_attributes(form_name, output_type='list')
print(f"Form: {form_name}")
print(f"Total supported attributes: {len(attrs)}")
print("Sample topological attributes:", [a for a in attrs if 'atom' in a or 'group' in a or 'bond' in a][:8])
print("Sample structural attributes:", [a for a in attrs if 'box' in a or 'structure' in a or 'coordinates' in a][:8])


Form: molsysmt.MolSys
Total supported attributes: 115
Sample topological attributes: ['atom_index', 'atom_name', 'atom_id', 'atom_type', 'group_index', 'group_name', 'group_id', 'group_type']
Sample structural attributes: ['structure_index', 'structure_id', 'structure_chemical_state_index', 'box', 'box_shape', 'box_angles', 'box_lengths', 'box_volume']


## Implemented Operations

The `molsysmt.MolSys` form adapter implements the full set of MolSysMT core operations:

| Operation | Function / Class | Description |
|---|---|---|
| **Get** | {func}`molsysmt.basic.get` | Extracts topological, structural, and mechanical attributes |
| **Set** | {func}`molsysmt.basic.set` | Modifies attributes in-place (coordinates, atom names, box dimensions) |
| **Extract** | {func}`molsysmt.basic.extract` | Filters a subsystem by selection or frame slice |
| **Copy** | {func}`molsysmt.basic.copy` | Creates an independent deep copy in memory |
| **Add** | {func}`molsysmt.basic.add` | Adds atoms or groups into an existing system |
| **Merge** | {func}`molsysmt.basic.merge` | Merges multiple molecular systems into a unified system |
| **Append Structures** | {func}`molsysmt.basic.append_structures` | Appends new frames or trajectories onto an existing topology |
| **Iterator** | {class}`molsysmt.basic.Iterator` | Iterates frame-by-frame or chunk-by-chunk over structures |
| **Bonds Manipulation** | `add_bonds` / `remove_bonds` | Dynamically defines or removes covalent bonds between atom pairs |

:::{note}
Each form in MolSysMT implements only the subset of operations relevant to its nature (for instance, topology-only forms omit coordinate setters or trajectory append operations).
:::


### Loading and Querying a MolSys Object

Let's load the bundled T4 lysozyme system (`181L`) into a `molsysmt.MolSys` instance:


In [3]:
# Load bundled system into molsysmt.MolSys
molsys = msm.convert(msm.systems['T4 lysozyme L99A']['181l.h5msm'], to_form='molsysmt.MolSys')

# Query topology and structure with msm.get
n_atoms, n_groups, n_structures = msm.get(molsys, n_atoms=True, n_groups=True, n_structures=True)
box = msm.get(molsys, box=True)

print(f"System: {n_atoms} atoms across {n_groups} residues and {n_structures} structure(s).")
print(f"Unit cell box shape: {box.shape}")


System: 1441 atoms across 302 residues and 1 structure(s).
Unit cell box shape: (1, 3, 3)


### Extracting Subsystems and Inspecting Properties

We can easily extract specific components or selections:


In [4]:
# Extract only protein atoms
protein_only = msm.extract(molsys, selection='molecule_type=="protein"')
n_protein_atoms = msm.get(protein_only, n_atoms=True)
print(f"Extracted protein subsystem: {n_protein_atoms} atoms.")

# Inspect coordinates
coords = msm.get(protein_only, coordinates=True)
print("Coordinates shape:", coords.shape)


Extracted protein subsystem: 1289 atoms.
Coordinates shape: (1, 1289, 3)


## Form Conversions (Interoperability Matrix)

`molsysmt.MolSys` provides direct, optimized bidirectional converters to and from a wide variety of class objects, file formats, and chemical strings:

### Supported Target Forms

| Target Form | Form Type | Converter Function | API Documentation | Transferred Information |
|---|---|---|---|---|
| `molsysmt.Topology` | `class` | `to_molsysmt_Topology` | {func}`~molsysmt.form.molsysmt_MolSys.to_molsysmt_Topology.to_molsysmt_Topology` | Full topology without coordinates |
| `molsysmt.Structures` | `class` | `to_molsysmt_Structures` | {func}`~molsysmt.form.molsysmt_MolSys.to_molsysmt_Structures.to_molsysmt_Structures` | Trajectory coordinates, box dimensions, and timestamps |
| `molsysmt.MolSysDict` | `class` | `to_molsysmt_MolSysDict` | {func}`~molsysmt.form.molsysmt_MolSys.to_molsysmt_MolSysDict.to_molsysmt_MolSysDict` | Python dictionary representation of the full system |
| `openmm.Topology` | `class` | `to_openmm_Topology` | {func}`~molsysmt.form.molsysmt_MolSys.to_openmm_Topology.to_openmm_Topology` | Native OpenMM topology representation |
| `openmm.System` | `class` | `to_openmm_System` | {func}`~molsysmt.form.molsysmt_MolSys.to_openmm_System.to_openmm_System` | Parameterized OpenMM System object |
| `openmm.Simulation` | `class` | `to_openmm_Simulation` | {func}`~molsysmt.form.molsysmt_MolSys.to_openmm_Simulation.to_openmm_Simulation` | Configured OpenMM Simulation ready for dynamics |
| `mdtraj.Trajectory` | `class` | `to_mdtraj_Trajectory` | {func}`~molsysmt.form.molsysmt_MolSys.to_mdtraj_Trajectory.to_mdtraj_Trajectory` | MDTraj trajectory container with topology |
| `mdtraj.Topology` | `class` | `to_mdtraj_Topology` | {func}`~molsysmt.form.molsysmt_MolSys.to_mdtraj_Topology.to_mdtraj_Topology` | MDTraj topology container |
| `parmed.Structure` | `class` | `to_parmed_Structure` | {func}`~molsysmt.form.molsysmt_MolSys.to_parmed_Structure.to_parmed_Structure` | ParmEd molecular structure |
| `pdbfixer.PDBFixer` | `class` | `to_pdbfixer_PDBFixer` | {func}`~molsysmt.form.molsysmt_MolSys.to_pdbfixer_PDBFixer.to_pdbfixer_PDBFixer` | PDBFixer instance for structure cleaning/preparation |
| `rdkit.Mol` | `class` | `to_rdkit_Mol` | {func}`~molsysmt.form.molsysmt_MolSys.to_rdkit_Mol.to_rdkit_Mol` | RDKit chemical molecule with bond orders |
| `nglview.NGLWidget` | `class` | `to_nglview_NGLWidget` | {func}`~molsysmt.form.molsysmt_MolSys.to_nglview_NGLWidget.to_nglview_NGLWidget` | Interactive 3D visualization widget |
| `networkx.Graph` | `class` | `to_networkx_Graph` | {func}`~molsysmt.form.molsysmt_MolSys.to_networkx_Graph.to_networkx_Graph` | Connectivity network graph |
| `biopython.Seq` | `class` | `to_biopython_Seq` | {func}`~molsysmt.form.molsysmt_MolSys.to_biopython_Seq.to_biopython_Seq` | BioPython sequence object |
| `biopython.SeqRecord` | `class` | `to_biopython_SeqRecord` | {func}`~molsysmt.form.molsysmt_MolSys.to_biopython_SeqRecord.to_biopython_SeqRecord` | BioPython sequence record |
| `file:h5msm` | `file` | `to_file_h5msm` | {func}`~molsysmt.form.molsysmt_MolSys.to_file_h5msm.to_file_h5msm` | Binary HDF5 serialized file |
| `file:pdb` | `file` | `to_file_pdb` | {func}`~molsysmt.form.molsysmt_MolSys.to_file_pdb.to_file_pdb` | Standard PDB coordinate file |
| `file:psf` | `file` | `to_file_psf` | {func}`~molsysmt.form.molsysmt_MolSys.to_file_psf.to_file_psf` | CHARMM/NAMD PSF topology file |
| `file:molsys_yaml` | `file` | `to_file_molsys_yaml` | {func}`~molsysmt.form.molsysmt_MolSys.to_file_molsys_yaml.to_file_molsys_yaml` | YAML human-readable specification file |
| `string:pdb_text` | `string` | `to_string_pdb_text` | {func}`~molsysmt.form.molsysmt_MolSys.to_string_pdb_text.to_string_pdb_text` | PDB format text block in memory |
| `string:amino_acids_1` | `string` | `to_string_amino_acids_1` | {func}`~molsysmt.form.molsysmt_MolSys.to_string_amino_acids_1.to_string_amino_acids_1` | Primary protein sequence (1-letter codes) |
| `string:amino_acids_3` | `string` | `to_string_amino_acids_3` | {func}`~molsysmt.form.molsysmt_MolSys.to_string_amino_acids_3.to_string_amino_acids_3` | Primary protein sequence (3-letter codes) |


### Conversion Demonstrations

Here is an example converting `molsysmt.MolSys` to a sequence string and to a topology object:


In [5]:
# Convert MolSys to 1-letter amino acid sequence string
seq_1 = msm.convert(protein_only, to_form='string:amino_acids_1')
print(f"Sequence (1-letter, first 50 residues):\n{seq_1[:50]}...")

# Convert MolSys to molsysmt.Topology
topology_only = msm.convert(molsys, to_form='molsysmt.Topology')
print(f"Converted topology object: {topology_only}")


Sequence (1-letter, first 50 residues):
MNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKSELDKAI...
Converted topology object: <molsysmt.native.topology.Topology object at 0x736c0cc6cb90>


:::{seealso} Related Tools & References
:class: dropdown

- {ref}`Convert <Tutorial_Convert>`: Converting between molecular system representations with {func}`molsysmt.basic.convert`.
- {ref}`Get <Tutorial_Get>`: Querying attributes from any form with {func}`molsysmt.basic.get`.
- {ref}`Set <Tutorial_Set>`: Modifying attributes with {func}`molsysmt.basic.set`.
- {ref}`Extract <Tutorial_Extract>`: Subsystem extraction with {func}`molsysmt.basic.extract`.
- {ref}`Iterator <Tutorial_Iterator>`: Trajectory streaming with {class}`molsysmt.basic.Iterator`.
- {ref}`Topology <Tutorial_Form_molsysmt_Topology>`: Native topology-only form.
- {ref}`Structures <Tutorial_Form_molsysmt_Structures>`: Native trajectory coordinates form.
:::
